# 04 · Stage 3 — Physics3DEnvironment

**Question:** does the same Agent / API design survive the move to 3D?

The action grows to `(dx, dy, dz)`, the physics core is called with 3-vectors,
and a pinhole camera projects depth-sorted billboards.  `BaseEnvironment`,
`AttackAgent`, `AttackReward` and the experiment driver are untouched.

In [ ]:
# Section 1: Setup
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "configs" / "default.yaml").exists():
    ROOT = ROOT.parent          # running from notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from configs.loader import load_config, build_victim, resolve

cfg = load_config()
print("project root:", ROOT)
print("victim config:", cfg["victim"])

Depth now matters: occluding the target requires getting *in front of* it, and
an object further from the camera than the target has no effect at all.

In [ ]:
from configs.loader import build_stage3_env
from environments.sealed import seal

victim = build_victim(cfg)
env = build_stage3_env(cfg, victim, obstacles=False)
api = seal(env)

print(api.action_space().describe())
print()
print(api.observation_space().describe())

## Section 2: Environment

The agent asks for a *push*, not a position.  Constraint check, physics,
collision and rendering all happen behind `step()`.

In [ ]:
obs = api.reset(seed=0)
base = env.pop_telemetry()
print("clean scene:", base["baseline_class"], f"{base['baseline_confidence']:.3f}")

for _ in range(5):
    obs, reward, terminated, truncated, info = api.step(np.ones(api.action_space().n) * 0.8)
    print(f"reward={reward:+.4f}  valid={info['action_valid']}  terminated={terminated}")

## Section 3: Visualization

In [ ]:
env_obs = build_stage3_env(cfg, victim, obstacles=True)
env_obs.reset(seed=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(env.clean_image());   axes[0].set_title("clean scene (no attacker)")
axes[1].imshow(env.render_human());  axes[1].set_title("after 5 pushes")
axes[2].imshow(env_obs.render_human()); axes[2].set_title("obstacle variant")
for ax in axes: ax.axis("off")
plt.show()

## Section 4: Baseline — Random and Greedy

Exactly the agents from Stage 1, unchanged.

In [ ]:
from agents.random_agent import RandomAgent
from agents.greedy_agent import GreedyAgent
from evaluation.runner import run_episodes
from evaluation.metrics import summarize
from evaluation.report import format_table

EPISODES = 8        # the script uses cfg[stage]["eval_episodes"]

summaries, conf_curves = [], {}
for name, agent in [("random", RandomAgent(seed=0)), ("greedy", GreedyAgent(seed=0))]:
    records, traces = run_episodes(env, agent, EPISODES, method=name, seed=0)
    s = summarize(records); s["label"] = name
    summaries.append(s)
    conf_curves[name] = [t.confidences for t in traces]

print(format_table(summaries))

## Section 5: Attack Agent — PPO


The same `PPOAgent`, the same hyper-parameters, one extra action dimension.

In [ ]:
from pathlib import Path
from agents.ppo_agent import PPOAgent

model_path = resolve(cfg["output"]["results_dir"]) / "stage3" / "ppo_no_obstacle.zip"
TRAIN_HERE = False        # flip to True to train inside the notebook (slow on CPU)

if model_path.exists() and not TRAIN_HERE:
    ppo = PPOAgent.load(model_path)
    print("loaded", model_path)
else:
    ppo = PPOAgent.train(
        env_factory=lambda: build_stage3_env(cfg, victim, obstacles=False),
        total_timesteps=2000,       # a demo budget; scripts use cfg[...]["ppo"]
        seed=0,
    )
    print("trained a short demo policy")

In [ ]:
records, traces = run_episodes(env, ppo, EPISODES, method="ppo", seed=0)
s = summarize(records); s["label"] = "ppo"
summaries.append(s)
conf_curves["ppo"] = [t.confidences for t in traces]
print(format_table(summaries))

## Section 6: Evaluation

In [ ]:
import json
summary_file = resolve(cfg["output"]["results_dir"]) / "stage3" / "summary.json"
if summary_file.exists():
    import pandas as pd
    full = pd.DataFrame(json.loads(summary_file.read_text()))
    display(full[["label", "attack_success_rate", "mean_confidence_drop",
                  "mean_reward", "mean_movement_cost", "mean_episode_length"]].round(3))
else:
    print("run scripts/run_stage3.py for the full-budget numbers")

## Section 7: Visualization

In [ ]:
from evaluation.plots import PALETTE

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, episodes in conf_curves.items():
    length = max(len(e) for e in episodes)
    padded = np.array([e + [e[-1]] * (length - len(e)) for e in episodes], dtype=float)
    axes[0].plot(np.arange(1, length + 1), padded.mean(0), label=name,
                 color=PALETTE.get(name), linewidth=2)
axes[0].axhline(base["baseline_confidence"], ls="--", c="k", lw=1, label="clean")
axes[0].set_xlabel("step"); axes[0].set_ylabel("victim confidence")
axes[0].grid(alpha=.3); axes[0].legend(); axes[0].set_title("confidence during an episode")

names = [s["label"] for s in summaries]
axes[1].bar(names, [s["attack_success_rate"] for s in summaries],
            color=[PALETTE.get(n, "#888") for n in names])
axes[1].set_title("attack success rate"); axes[1].grid(axis="y", alpha=.3)
plt.show()

In [ ]:
from evaluation.plots import draw_detections

_, demo = run_episodes(env, ppo, 1, method="ppo", seed=123, collect_frames=True)
frame = demo[0].best_frame()
if frame is not None:
    plt.figure(figsize=(6, 6))
    plt.imshow(draw_detections(frame, env.victim_report(frame), highlight=base["baseline_class"]))
    plt.axis("off"); plt.title(f"best PPO frame: confidence {min(demo[0].confidences):.3f}")
    plt.show()

Full experiment (both variants, full budgets, all figures):

```
uv run python scripts/run_stage3.py
```